# Practice 32 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

---
## Level 1 — `.loc` and `.iloc` under pressure

### Task 1: penguins — index manipulation

Load `sns.load_dataset('penguins')`. Drop nulls.

1. Set `species` as the index. Use `.loc['Adelie']` to get all Adelie penguins. How many are there?
2. Reset the index (use `reset_index()`). Now sort by `body_mass_g` descending. What does `df.loc[0]` return vs `df.iloc[0]`? Are they the same?
3. Use `.iloc[0:5, 2:5]` to get the first 5 rows and columns 3–5 by position. Then do the same with `.loc` using column names — confirm the results match.
4. Filter to only `'Dream'` island penguins using `.loc`. On that result, use `.iloc[-1]` to get the last row.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

In [ ]:
# Your code here
penguins = sns.load_dataset('penguins').dropna()

penguins = penguins.set_index('species')
a = penguins.loc['Adelie']
a.shape

penguins =penguins.reset_index().sort_values('body_mass_g')
penguins.loc[0]
penguins.iloc[0]

### different, .loc returns the first row before sorting .iloc returns after sorting first row
p = penguins.iloc[0:5, 2:5]
p1 = penguins.loc[penguins.index[0:5],['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']]
p.equals(p1)

#print(p)
#print(p1)

penguins.loc[penguins['island']=='Dream'].iloc[-1]


True

---
## Level 2 — Open questions

### Task 2: diamonds — price analysis with category

Load `sns.load_dataset('diamonds')`. Apply ordered dtypes to `cut` (`Fair < Good < Very Good < Premium < Ideal`) and `color` (`D < E < F < G < H < I < J`).

Five questions — no sub-steps:

1. What fraction of diamonds have both cut `>= 'Premium'` and color `<= 'F'`?
2. Among `'Ideal'` cut diamonds only, which color has the highest mean price? Use `.loc` to filter, `np.argmax()` for the answer.
3. Add `price_per_carat`. Use `.agg()` with a dict of tuples to get: mean price, median price_per_carat, and total count — for each cut level. Which cut has the highest median price_per_carat?
4. Build a pivot table of mean `price_per_carat` by `cut` × `color` with `observed=True`. Use `.loc` to extract just the `'Ideal'` row.
5. After filtering to diamonds cheaper than $1000, call `.cat.remove_unused_categories()` on `cut`. Which cuts disappeared?

In [62]:
# Your code here

diamonds = sns.load_dataset('diamonds')
co = pd.CategoricalDtype(categories=['Fair','Good','Very Good','Premium','Ideal'],
                         ordered=True)
clo = pd.CategoricalDtype(categories= ['D','E','F','G','H','I','J'],
                          ordered=True)
diamonds['cut'] = diamonds['cut'].astype(co)
diamonds['color'] = diamonds['color'].astype(clo)

f = (diamonds['cut']>='Premium') & (diamonds['color']<='F')
f.mean()

fi = diamonds.loc[diamonds['cut']=='Ideal']
m = fi.groupby('color', observed=True)['price'].mean()
m.index[np.argmax(m)]


diamonds['price_per_carat'] = diamonds['price']/diamonds['carat']

t =diamonds.groupby('cut', observed=True).agg(
    mean_price = ('price','mean'),
    median_price_per_carat = ('price_per_carat', 'median'),
    total_count = ('price','count')
)

t['median_price_per_carat'].idxmax()

pv = diamonds.pivot_table(
    index= 'cut',
    columns='color',
    values = 'price_per_carat',
    observed=True
)

pv.loc['Ideal']
fil = diamonds[diamonds['price']<1000].copy()
fil['cut'] = fil['cut'].cat.remove_unused_categories()
fil['cut'].cat.categories
### all are preserved

Index(['Fair', 'Good', 'Very Good', 'Premium', 'Ideal'], dtype='object')

---
## Level 3 — Aggregation without pipe

### Task 3: titanic — grouped survival audit

Load `sns.load_dataset('titanic')`. No `.pipe()` in this task.

- Convert `pclass` to ordered category (`3 < 2 < 1`) and `sex`, `embarked` to unordered category.
- Use `.agg()` with a dict of tuples to produce a summary per `pclass`: mean survival rate, mean fare, median age. Use `.loc` to extract only first and second class rows from the result.
- Build a pivot table: survival rate by `embarked` × `pclass`, `observed=True`. Stack it. Use `.xs()` with a tuple to extract one specific `(embarked, pclass)` combination.
- Among passengers who embarked at `'C'` (Cherbourg), use `.loc` to filter them. What is the survival rate by sex for this group? Use `groupby` with `observed=True`.

In [76]:
# Your code here

titanic = sns.load_dataset('titanic')
po = pd.CategoricalDtype(categories=[3,2,1],ordered=True)
titanic['pclass'] = titanic['pclass'].astype(po)
titanic[['sex','embarked']] = titanic[['sex', 'embarked']].astype('category')

s = titanic.groupby('pclass',observed=True).agg(
    mean_surv_rate = ('survived','mean'),
    mean_fare = ('fare','mean'),
    median_age = ('age','median')
)
s.loc[[1,2]]

pv = titanic.pivot_table(
    index = 'embarked',
    columns = 'pclass',
    values = 'survived',
    observed=True
)
pv
pv.stack().xs(('C',3))


fc = titanic.loc[titanic['embarked'] == 'C']

fc.groupby('sex',observed=True)['survived'].mean()

sex
female    0.876712
male      0.305263
Name: survived, dtype: float64

---
## Level 4 — Full pipeline

### Task 4: mpg — pipe chain and reshape

Load `sns.load_dataset('mpg')`. Drop rows where `horsepower` is null.

Write a `.pipe()` chain with 3 functions — write them yourself:
- One converts `cylinders` (ordered: `3 < 4 < 5 < 6 < 8`) and `origin` (unordered) to category
- One adds `power_to_weight` (`horsepower / weight`) and a cylinder-group mean `power_to_weight` using `transform`
- One classifies each car into `'efficient'` (mpg > 30), `'average'` (mpg 20–30), or `'inefficient'` (mpg < 20) using `.apply()`

After the chain:
- Use `.agg()` with a dict of tuples: mean `mpg`, mean `power_to_weight`, total count — per `origin`. Use `observed=True`.
- Build a pivot table: count of cars by `efficiency_class` × `origin`, `observed=True`. Which origin has the most efficient cars?
- Use `.loc` to extract only `'inefficient'` cars. What is their mean `horsepower`?

In [87]:
# Your code here

mpg = sns.load_dataset('mpg').dropna(subset='horsepower')

def co(df):
    coo = pd.CategoricalDtype(categories=[3,4,5,6,8],ordered=True)
    df['cylinders'] = df['cylinders'].astype(coo)
    df['origin'] = df['origin'].astype('category')
    return(df)

def adp(df):
    df['power_to_weight'] = df['horsepower']/df['weight']
    df['cmp'] = df.groupby('cylinders',observed = True)['power_to_weight'].transform('mean')
    return df 

def ce(df):
    df['efficiency_class'] = df['mpg'].apply(lambda x: 'efficient' if x >30
                                  else 'average' if x>20
                                  else 'inefficient')
    return(df)

mpg = mpg.pipe(co).pipe(adp).pipe(ce)

s = mpg.groupby('origin', observed=True).agg(
    mean_mpg = ('mpg','mean'),
    mean_ptw = ('power_to_weight', 'mean'),
    total_count = ('mpg','count')
)

pv = mpg.pivot_table(
    index = 'efficiency_class',
    columns = 'origin',
    values = 'mpg',
    aggfunc = 'count',
    observed = True
)

pv.loc['efficient'].idxmax()

fil = mpg.loc[mpg['efficiency_class']=='inefficient']
fil['horsepower'].mean()

np.float64(137.45)

After the chain:
- Use `.agg()` with a dict of tuples: mean `mpg`, mean `power_to_weight`, total count — per `origin`. Use `observed=True`.
- Build a pivot table: count of cars by `efficiency_class` × `origin`, `observed=True`. Which origin has the most efficient cars?
- Use `.loc` to extract only `'inefficient'` cars. What is their mean `horsepower`?